# Netflix Subscription Lifecycle & Engagement Optimization: reproducible analysis

Companion notebook to the project [README](../README.md). Every number quoted in the README is produced by a cell below, so each claim can be checked. Run the cells top to bottom.

**Conventions**

- *Churned* is the `churned` flag (`Yes` = churned). *MRR* is the sum of `monthly_fee`.
- Group comparisons of churn use chi-square tests. Comparisons of numeric behavior use Mann-Whitney, reported as AUC (0.5 means no separation between churned and active users).
- With 19 hypothesis tests, p-values are also reported with a Holm correction. A nominal p-value below 0.05 is only called a finding if it survives the correction.
- Random seed is 42 everywhere.

**Sections**

1. Data quality
2. Baseline churn and revenue exposure
3. Does anything separate churned users from active ones?
4. Predictive check
5. Apparent hot spots (heatmap, country, genre)
6. Segmentation
7. Statistical power
8. Acquisition trend artifact
9. Sizing

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 140)
SEED = 42


def show(frame, fmt="{:,.3f}"):
    # Display a DataFrame with a fixed number format (keeps p-values readable)
    with pd.option_context("display.float_format", fmt.format):
        display(frame)


# Works whether the notebook is opened from the repository root or from analysis/
DATA = next(p for p in (Path("netflix_user_behavior_dataset.csv"),
                        Path("../netflix_user_behavior_dataset.csv")) if p.exists())

df = pd.read_csv(DATA)
RAW_SHAPE = df.shape
df["churn"] = (df["churned"] == "Yes").astype(int)
MRR = df["monthly_fee"].sum()

# Same 10-year age bins as the heatmap (bin label = lower bound; the "10" bin holds ages 18-19)
df["age_bin"] = df["age"] // 10 * 10

print(f"Loaded {RAW_SHAPE[0]:,} rows x {RAW_SHAPE[1]} columns from {DATA}")

Loaded 50,000 rows x 20 columns from ../netflix_user_behavior_dataset.csv


## 1. Data quality

In [2]:
checks = pd.DataFrame({
    "check": ["Rows x columns", "Duplicate user_id", "Duplicate rows (ignoring user_id)", "Missing values",
              "Age range", "Account age range (months)", "Days since last login range",
              "Average watch time range (minutes)"],
    "result": [f"{RAW_SHAPE[0]:,} x {RAW_SHAPE[1]}", df.user_id.duplicated().sum(),
               df.drop(columns="user_id").duplicated().sum(), int(df.isna().sum().sum()),
               f"{df.age.min()} to {df.age.max()}",
               f"{df.account_age_months.min()} to {df.account_age_months.max()}",
               f"{df.days_since_last_login.min()} to {df.days_since_last_login.max()}",
               f"{df.avg_watch_time_minutes.min()} to {df.avg_watch_time_minutes.max()}"],
})
checks

,check,result
0,Rows x columns,"50,000 x 20"
1,Duplicate user_id,0
2,Duplicate rows (ignoring user_id),0
3,Missing values,0
4,Age range,18 to 64
5,Account age range (months),1 to 59
6,Days since last login range,0 to 59
7,Average watch time range (minutes),10 to 299


In [3]:
# Are the numeric fields flat? Chi-square goodness of fit against a discrete uniform distribution.
rows = []
for col in ["age", "account_age_months", "avg_watch_time_minutes", "watch_sessions_per_week",
            "binge_watch_sessions", "completion_rate", "content_interactions",
            "recommendation_click_rate", "days_since_last_login"]:
    counts = df[col].value_counts().reindex(np.arange(df[col].min(), df[col].max() + 1), fill_value=0)
    rows.append({"field": col, "distinct_values": len(counts), "min_count": counts.min(),
                 "max_count": counts.max(), "uniform_gof_p": stats.chisquare(counts).pvalue})
uniformity = pd.DataFrame(rows)
show(uniformity)

balance = {c: round(stats.chisquare(df[c].value_counts()).pvalue, 3)
           for c in ["gender", "country", "payment_method", "primary_device", "favorite_genre"]}
print("Category balance, chi-square goodness of fit p-values:", balance)

,field,distinct_values,min_count,max_count,uniform_gof_p
0,age,47,982,1144,0.236
1,account_age_months,59,791,907,0.770
2,avg_watch_time_minutes,290,137,210,0.259
3,watch_sessions_per_week,19,2531,2728,0.459
4,binge_watch_sessions,15,3224,3407,0.523
5,completion_rate,70,659,765,0.961
6,content_interactions,50,914,1098,0.009
7,recommendation_click_rate,100,438,563,0.375
8,days_since_last_login,60,737,910,0.074


Category balance, chi-square goodness of fit p-values: {'gender': np.float64(0.302), 'country': np.float64(0.436), 'payment_method': np.float64(0.972), 'primary_device': np.float64(0.417), 'favorite_genre': np.float64(0.89)}


In [4]:
# Does the monthly fee follow the plan?
print("Fee values:", df.monthly_fee.value_counts().sort_index().to_dict())
print("\nMean fee by plan:", df.groupby("subscription_type")["monthly_fee"].mean().round(2).to_dict())
print(f"Kruskal-Wallis, fee ~ plan: p = {stats.kruskal(*[g.monthly_fee for _, g in df.groupby('subscription_type')]).pvalue:.3f}")
pd.crosstab(df.subscription_type, df.monthly_fee, normalize="index").mul(100).round(1)

Fee values: {7.99: 16743, 12.99: 16463, 15.99: 16794}

Mean fee by plan: {'Basic': 12.32, 'Premium': 12.31, 'Standard': 12.33}
Kruskal-Wallis, fee ~ plan: p = 0.877


monthly_fee,7.99,12.99,15.99
subscription_type,,,
Basic,33.60,32.90,33.50
Premium,33.70,32.70,33.60
Standard,33.30,33.10,33.60


**Takeaway.** No duplicates and no missing values. Nearly every numeric field is flat across its range (age 18 to 64, account age 1 to 59 months, days since last login 0 to 59), and categories are evenly balanced, which is typical of generated data. The monthly fee takes three values (7.99, 12.99, 15.99) but is unrelated to the plan label, so a "Premium" user is as likely to pay 7.99 as 15.99.

## 2. Baseline churn and revenue exposure

In [5]:
lost_mrr = df.loc[df.churn == 1, "monthly_fee"].sum()
print(f"Churned users: {df.churn.sum():,} of {len(df):,} = {df.churn.mean() * 100:.2f}%")
print(f"Total MRR: ${MRR:,.0f}")
print(f"MRR of churned users: ${lost_mrr:,.0f} = {lost_mrr / MRR * 100:.2f}% of MRR")
print(f"Average fee, churned vs active: ${df[df.churn == 1].monthly_fee.mean():.2f} vs ${df[df.churn == 0].monthly_fee.mean():.2f}")

plan = df.groupby("subscription_type").agg(users=("churn", "size"), churned=("churn", "sum"),
                                           churn_rate=("churn", "mean"), mrr=("monthly_fee", "sum"))
plan["churn_rate"] *= 100
plan

Churned users: 9,964 of 50,000 = 19.93%
Total MRR: $616,167
MRR of churned users: $122,380 = 19.86% of MRR
Average fee, churned vs active: $12.28 vs $12.33


,users,churned,churn_rate,mrr
subscription_type,,,,
Basic,14873,2861,19.24,"183,206.27"
Premium,15196,3099,20.39,"187,125.04"
Standard,19931,4004,20.09,"245,835.69"


**Takeaway.** 19.9% of users have churned, taking $122K (19.9%) of the $616K MRR with them. Churned and active users pay the same on average, so revenue churn equals user churn.

## 3. Does anything separate churned users from active ones?

Eight categorical dimensions (chi-square) and eleven numeric fields (Mann-Whitney), 19 tests in total, with a Holm correction.

In [6]:
cat_rows = []
for col in ["subscription_type", "gender", "country", "favorite_genre", "primary_device",
            "payment_method", "age_bin", "devices_used"]:
    rates = df.groupby(col)["churn"].mean() * 100
    p = stats.chi2_contingency(pd.crosstab(df[col], df.churn))[1]
    cat_rows.append({"variable": col, "lowest_churn_%": rates.min(), "highest_churn_%": rates.max(), "p": p})
cat_tests = pd.DataFrame(cat_rows)

num_rows = []
for col in ["age", "account_age_months", "avg_watch_time_minutes", "watch_sessions_per_week",
            "binge_watch_sessions", "completion_rate", "rating_given", "content_interactions",
            "recommendation_click_rate", "days_since_last_login", "monthly_fee"]:
    churned, active = df.loc[df.churn == 1, col], df.loc[df.churn == 0, col]
    u = stats.mannwhitneyu(churned, active)
    num_rows.append({"variable": col, "churned_mean": churned.mean(), "active_mean": active.mean(),
                     "AUC": u.statistic / (len(churned) * len(active)), "p": u.pvalue})
num_tests = pd.DataFrame(num_rows)

all_p = list(cat_tests.p) + list(num_tests.p)
holm = multipletests(all_p, method="holm")[1]
cat_tests["holm_p"] = holm[:len(cat_tests)]
num_tests["holm_p"] = holm[len(cat_tests):]

show(cat_tests)
show(num_tests)

,variable,lowest_churn_%,highest_churn_%,p,holm_p
0,subscription_type,19.236,20.394,0.033,0.619
1,gender,19.591,20.298,0.271,1.000
2,country,19.245,20.963,0.261,1.000
3,favorite_genre,19.262,20.505,0.626,1.000
4,primary_device,19.583,20.289,0.493,1.000
5,payment_method,19.661,20.180,0.752,1.000
6,age_bin,19.429,20.288,0.762,1.000
7,devices_used,19.828,20.047,0.880,1.000


,variable,churned_mean,active_mean,AUC,p,holm_p
0,age,40.914,40.996,0.498,0.593,1.000
1,account_age_months,29.640,29.932,0.495,0.126,1.000
2,avg_watch_time_minutes,154.104,155.156,0.496,0.263,1.000
3,watch_sessions_per_week,9.948,9.997,0.497,0.435,1.000
4,binge_watch_sessions,6.974,7.010,0.498,0.459,1.000
5,completion_rate,64.459,64.552,0.499,0.683,1.000
6,rating_given,3.010,3.000,0.502,0.466,1.000
7,content_interactions,24.112,24.355,0.495,0.135,1.000
8,recommendation_click_rate,49.432,49.596,0.498,0.615,1.000
9,days_since_last_login,29.426,29.407,0.500,0.921,1.000


In [7]:
print(f"Tests run: {len(all_p)}")
print(f"Nominal p < 0.05: {[(v, round(p, 3)) for v, p in zip(list(cat_tests.variable) + list(num_tests.variable), all_p) if p < 0.05]}")
print(f"Expected false positives at the 0.05 level: {0.05 * len(all_p):.1f}")
print(f"Smallest Holm-adjusted p: {min(holm):.3f}")
print(f"AUC range across the numeric fields: {num_tests.AUC.min():.3f} to {num_tests.AUC.max():.3f}")

Tests run: 19
Nominal p < 0.05: [('subscription_type', 0.033)]
Expected false positives at the 0.05 level: 1.0
Smallest Holm-adjusted p: 0.619
AUC range across the numeric fields: 0.495 to 0.502


**Takeaway.** One of 19 tests is nominally significant (churn by plan, p = 0.033), which is what chance alone produces at this threshold. After correction its p-value is 0.62. Every numeric field has an AUC between 0.495 and 0.502: churned and active users are indistinguishable on age, tenure, watch time, sessions, completion, interactions, click rate, rating and recency.

## 4. Predictive check

If a combination of fields carried signal, a model would find it. Five-fold cross-validated AUC using all fields at once.

In [8]:
X = pd.get_dummies(df.drop(columns=["user_id", "churned", "churn", "age_bin"]), drop_first=True).astype(float)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=SEED),
}
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, df.churn, cv=cv, scoring="roc_auc")
    results[name] = {"cv_auc_mean": scores.mean(), "cv_auc_sd": scores.std()}
show(pd.DataFrame(results).T)

,cv_auc_mean,cv_auc_sd
Logistic regression,0.501,0.003
Gradient boosting,0.498,0.004


**Takeaway.** Both models score an AUC of about 0.50, no better than a coin flip. No combination of the available fields predicts churn.

## 5. Apparent hot spots

The heatmap (age by plan), the map (country) and the treemap (genre) all show differences. Are they larger than chance?

### 5a. Age by plan (18 cells)

In [9]:
cells = df.groupby(["subscription_type", "age_bin"])["churn"].agg(rate="mean", users="size").reset_index()
print(f"Cells: {len(cells)}")
print(f"Churn rate range: {cells.rate.min() * 100:.2f}% to {cells.rate.max() * 100:.2f}%  (spread {(cells.rate.max() - cells.rate.min()) * 100:.2f} points)")
print(f"Cell size range: {cells.users.min():,} to {cells.users.max():,} users")
print(f"Users by age bin: {df.groupby('age_bin').size().to_dict()}")
print(f"Chi-square across the 18 cells: p = {stats.chi2_contingency(pd.crosstab([df.subscription_type, df.age_bin], df.churn))[1]:.3f}")

extremes = cells.sort_values("rate")
display(pd.concat([extremes.head(3), extremes.tail(3)]).assign(rate=lambda d: d.rate * 100).round(2))

Cells: 18
Churn rate range: 18.04% to 21.36%  (spread 3.32 points)
Cell size range: 618 to 4,349 users
Users by age bin: {10: 2082, 20: 10556, 30: 10794, 40: 10760, 50: 10536, 60: 5272}
Chi-square across the 18 cells: p = 0.183


,subscription_type,age_bin,rate,users
1,Basic,20,18.04,3115
6,Premium,10,18.12,640
16,Standard,50,19.12,4194
11,Premium,60,21.22,1602
13,Standard,20,21.23,4240
12,Standard,10,21.36,824


In [10]:
# Permutation test: shuffle churn labels across users and measure the max-min spread across the same 18 cells.
codes, _ = pd.factorize(df.subscription_type + "_" + df.age_bin.astype(str))
sizes = np.bincount(codes)
observed = cells.rate.max() - cells.rate.min()

rng = np.random.default_rng(SEED)
labels = df.churn.to_numpy()
null_spread = np.empty(2000)
for i in range(2000):
    rates = np.bincount(codes, weights=rng.permutation(labels)) / sizes
    null_spread[i] = rates.max() - rates.min()

print(f"Observed spread: {observed * 100:.2f} points")
print(f"Spread expected by chance: median {np.median(null_spread) * 100:.2f}, 95th percentile {np.percentile(null_spread, 95) * 100:.2f} points")
print(f"Permutation p-value (spread >= observed): {(null_spread >= observed).mean():.3f}")

Observed spread: 3.32 points
Spread expected by chance: median 3.42, 95th percentile 5.38 points
Permutation p-value (spread >= observed): 0.542


### 5b. Country

In [11]:
country = df.groupby("country")["churn"].agg(rate="mean", users="size")
country["ci95_+/-_pts"] = 1.96 * np.sqrt(country.rate * (1 - country.rate) / country.users) * 100
country["rate"] *= 100
display(country.sort_values("rate", ascending=False).round(2))

print(f"Chi-square, churn ~ country: p = {stats.chi2_contingency(pd.crosstab(df.country, df.churn))[1]:.3f}")
aus = df.country == "Australia"
print(f"Australia {df[aus].churn.mean() * 100:.2f}% vs all others {df[~aus].churn.mean() * 100:.2f}%, "
      f"p = {stats.chi2_contingency(pd.crosstab(aus, df.churn))[1]:.3f}")

,rate,users,ci95_+/-_pts
country,,,
Australia,20.96,5004,1.13
Japan,20.81,4907,1.14
India,20.31,5028,1.11
France,20.21,4919,1.12
UK,19.90,4929,1.11
Canada,19.84,4959,1.11
Brazil,19.51,5116,1.09
Spain,19.28,5027,1.09
Germany,19.27,5024,1.09


Chi-square, churn ~ country: p = 0.261
Australia 20.96% vs all others 19.81%, p = 0.056


### 5c. Genre and lost revenue

In [12]:
genre = df.groupby("favorite_genre").agg(users=("churn", "size"), churn_rate=("churn", "mean"))
genre["churn_rate"] *= 100
genre["lost_mrr"] = df[df.churn == 1].groupby("favorite_genre")["monthly_fee"].sum()
genre["share_of_lost_mrr_%"] = genre.lost_mrr / genre.lost_mrr.sum() * 100
display(genre.sort_values("lost_mrr", ascending=False).round(2))

print(f"Largest / smallest lost MRR: {genre.lost_mrr.max() / genre.lost_mrr.min():.3f}")
print(f"Equal share would be {100 / len(genre):.1f}% of lost MRR per genre")
print(f"Chi-square, churn ~ genre: p = {stats.chi2_contingency(pd.crosstab(df.favorite_genre, df.churn))[1]:.3f}")

,users,churn_rate,lost_mrr,share_of_lost_mrr_%
favorite_genre,,,,
Horror,6223,20.34,"15,701.34",12.83
Romance,6282,20.20,"15,630.31",12.77
Thriller,6257,20.51,"15,587.17",12.74
Comedy,6259,20.07,"15,454.44",12.63
Sci-Fi,6189,19.92,"15,372.67",12.56
Documentary,6352,19.69,"15,161.49",12.39
Action,6235,19.26,"14,789.99",12.09
Drama,6203,19.43,"14,682.95",12.00


Largest / smallest lost MRR: 1.069
Equal share would be 12.5% of lost MRR per genre
Chi-square, churn ~ genre: p = 0.626


**Takeaway.** The 3.3-point spread in the heatmap is what shuffled labels produce (permutation p = 0.54), and its most extreme cells are the smallest (ages 18-19, 618 to 824 users). Country churn ranges from 19.2% to 21.0% with a per-country margin of error of about 1.1 points. Lost revenue by genre ranges only from $14.7K to $15.7K because genres have similar numbers of users and equal churn.

## 6. Segmentation

The segmentation chart has four segments. The rules below reproduce its counts exactly (33,025 / 9,964 / 5,201 / 1,810).

In [13]:
low_watch = df.avg_watch_time_minutes <= 60
inactive = df.days_since_last_login > 14
active_user = df.churn == 0

df["segment"] = np.select(
    [~active_user, active_user & ~low_watch, active_user & low_watch & inactive, active_user & low_watch & ~inactive],
    ["Lost Customer", "Super Fan", "At Risk", "Active User"], default="?")

rules = {"Lost Customer": "churned", "Super Fan": "not churned and avg watch time > 60 min",
         "At Risk": "not churned, watch time <= 60 min, last login > 14 days ago",
         "Active User": "not churned, watch time <= 60 min, last login <= 14 days ago"}
seg = df.groupby("segment").agg(users=("churn", "size"), mrr=("monthly_fee", "sum"))
seg["share_of_users_%"] = seg.users / len(df) * 100
seg["share_of_mrr_%"] = seg.mrr / MRR * 100
seg["rule"] = pd.Series(rules)
display(seg.sort_values("users", ascending=False).round(1))

print(f"Super Fan share of non-churned users: {seg.loc['Super Fan', 'users'] / (df.churn == 0).sum() * 100:.1f}%")
print(f"Share of all users with avg watch time <= 60 min: {low_watch.mean() * 100:.1f}%")

,users,mrr,share_of_users_%,share_of_mrr_%,rule
segment,,,,,
Super Fan,33025,"407,539.80",66.00,66.10,not churned and avg watch time > 60 min
Lost Customer,9964,"122,380.40",19.90,19.90,churned
At Risk,5201,"64,037.00",10.40,10.40,"not churned, watch time <= 60 min, last login ..."
Active User,1810,"22,209.90",3.60,3.60,"not churned, watch time <= 60 min, last login ..."


Super Fan share of non-churned users: 82.5%
Share of all users with avg watch time <= 60 min: 17.6%


In [14]:
# Does the At Risk profile actually identify users who churn?
# Apply the profile to ALL users (including those already churned) and compare churn rates.
profile = low_watch & inactive
print(f"Users matching the At Risk profile: {profile.sum():,}")
print(f"  churn rate, profile vs everyone else: {df[profile].churn.mean() * 100:.2f}% vs {df[~profile].churn.mean() * 100:.2f}%, "
      f"chi-square p = {stats.chi2_contingency(pd.crosstab(profile, df.churn))[1]:.3f}")
print(f"  low watch time alone:  {df[low_watch].churn.mean() * 100:.2f}% vs {df[~low_watch].churn.mean() * 100:.2f}%, "
      f"p = {stats.chi2_contingency(pd.crosstab(low_watch, df.churn))[1]:.3f}")
print(f"  inactivity alone:      {df[inactive].churn.mean() * 100:.2f}% vs {df[~inactive].churn.mean() * 100:.2f}%, "
      f"p = {stats.chi2_contingency(pd.crosstab(inactive, df.churn))[1]:.3f}")

Users matching the At Risk profile: 6,545
  churn rate, profile vs everyone else: 20.53% vs 19.84%, chi-square p = 0.193
  low watch time alone:  20.14% vs 19.88%, p = 0.596
  inactivity alone:      19.95% vs 19.85%, p = 0.807


**Takeaway.** "Super Fan" covers 82.5% of active users because 60 minutes is only the 18th percentile of watch time, so the label separates almost nobody. The At Risk profile (5,201 active users, $64K MRR) is a sensible pilot audience, but it is not a validated risk score: users matching it churn at 20.5% versus 19.8% for everyone else (p = 0.19).

## 7. Statistical power

How large would a churn difference have to be to be detected? Minimum detectable difference between two groups at 80% power, 5% significance, baseline churn of 19.9%.

In [15]:
def mde_points(p, n1, n2):
    se = np.sqrt(p * (1 - p) / n1 + p * (1 - p) / n2)
    return (1.96 + 0.84) * se * 100

p0 = df.churn.mean()
power = pd.DataFrame({
    "comparison": ["Two halves of the base", "One plan vs the rest (about 15K users)",
                   "One country vs the rest (about 5K users)", "A group the size of At Risk (5.2K)",
                   "The smallest heatmap cell (about 620 users)"],
    "group_size": [25000, 15000, 5000, 5201, 620],
})
power["min_detectable_gap_pts"] = [mde_points(p0, n, len(df) - n) for n in power.group_size]
power.round(2)

,comparison,group_size,min_detectable_gap_pts
0,Two halves of the base,25000,1.00
1,One plan vs the rest (about 15K users),15000,1.09
2,One country vs the rest (about 5K users),5000,1.67
3,A group the size of At Risk (5.2K),5201,1.64
4,The smallest heatmap cell (about 620 users),620,4.52


**Takeaway.** A gap of about 1 point between plans, or 1.7 points for a country-sized group, would have been detected. The observed differences are smaller than that, so any real effect is small. Only the smallest heatmap cells (about 620 users) are too small to read, which is where the most extreme colors come from.

## 8. Acquisition trend artifact

The dataset has no registration date. The acquisition chart reconstructs one from `account_age_months`. Account ages are uniform from 1 to 59 months, and 59 is not a multiple of 12, so calendar months are not equally represented.

In [16]:
age_counts = df.account_age_months.value_counts().sort_index()
print(f"Users per account-age value: min {age_counts.min()}, max {age_counts.max()}; "
      f"uniform goodness of fit p = {stats.chisquare(age_counts).pvalue:.3f}")

# A registration month is (snapshot month - account age). Ages 12, 24, 36, 48 map back to the snapshot month
# itself, so that calendar month gets 4 cohorts of account age, while every other month gets 5.
by_residue = df.account_age_months.mod(12).value_counts().sort_index()
others = by_residue.drop(0).mean()
print(f"\nAccount ages that are multiples of 12 (1-59): {[a for a in range(1, 60) if a % 12 == 0]}")
print(f"Users in the 4-cohort month: {by_residue[0]:,}")
print(f"Average users in each of the other 11 months: {others:,.0f}")
print(f"Ratio: {by_residue[0] / others:.3f}  (expected 4/5 = 0.800)")

Users per account-age value: min 791, max 907; uniform goodness of fit p = 0.770

Account ages that are multiples of 12 (1-59): [12, 24, 36, 48]
Users in the 4-cohort month: 3,378
Average users in each of the other 11 months: 4,238
Ratio: 0.797  (expected 4/5 = 0.800)


**Takeaway.** One calendar month has 20% fewer users purely because of how the dates are reconstructed (3,378 versus about 4,238 for the others). That is the January dip in the acquisition chart. The rest of the line is flat, so no acquisition trend can be read from this field.

## 9. Sizing

What is a point of churn worth?

In [17]:
one_point = MRR * 0.01
print(f"Total MRR: ${MRR:,.0f}")
print(f"One percentage point of churn = ${one_point:,.0f} MRR = ${one_point * 12:,.0f} per year")

at_risk = df[df.segment == "At Risk"]
print(f"\nAt Risk pilot audience: {len(at_risk):,} users, ${at_risk.monthly_fee.sum():,.0f} MRR "
      f"({at_risk.monthly_fee.sum() / MRR * 100:.1f}% of total)")

Total MRR: $616,167
One percentage point of churn = $6,162 MRR = $73,940 per year

At Risk pilot audience: 5,201 users, $64,037 MRR (10.4% of total)


**Takeaway.** Each point of churn is worth about $6.2K of monthly revenue, or $74K a year. The At Risk audience holds $64K of MRR.